# Standalone magnetic solver (TRUST-LiMe) — validation


## Introduction

 Validation of the **standalone magnetic (induction) equation** added by TRUST-LiMe.

 The equation solved is

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

with the magnetic diffusivity $\eta$ and the transporting velocity $u$ supplied
as data (no Navier–Stokes coupling in this first phase).

 Every case is **self-contained** (inline `Mailler`/`Pave` mesh, no `.med` file)
 and records point probes (`Postraitement/Sondes`) into `.son` files which are
 compared below against closed-form solutions.

 | case | what it validates | reference |
|---|---|---|
| `invariance` | uniform $b$, $u=0$ — no spurious source | exact invariant (machine precision) |
| `diffusion` / `diffusion_refined` | Fourier-mode decay, $\eta$ scaling | $e^{-\eta\pi^2t}$ + mesh convergence |
| `advection` | convection operator, `vitesse_magnetique` wiring | peak conservation + transport direction |
| `vector_components` | multi-component (vector) unknown, per-component rates | two distinct eigenmodes |
| `steady_state` | equilibrium preservation, **non-uniform** $\eta$ | exact steady state |
| `varying_velocity` | **spatially varying** velocity transfer onto faces | exact invariant |
| `error_*` | malformed inputs fail cleanly | non-zero exit + readable message |


In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")

## Helpers: read a TRUST probe (`.son`) file

 Cases are copied by `trustutils` into `run.BUILD_DIRECTORY/<dataset>/`, where the
 `.son` probe files are written.


In [ ]:
import numpy as np
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, name):
    """Return (time, values) for probe `name` of case `case` (dataset name)."""
    path = f"{BUILD}/{case}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def ratio_error(case, tags, ref):
    """Max |b(t)/b(0) - ref(t)| over the given probes and all recorded times."""
    worst = 0.0
    for tag in tags:
        t, b = read_son(case, tag)
        if b[0] == 0:
            continue
        worst = max(worst, float(np.max(np.abs(b / b[0] - ref(t)))))
    return worst

print("BUILD =", BUILD)

## 1. Positive cases — run


In [ ]:
from trustutils import run

run.reset()
POSITIVE = ["invariance", "diffusion", "diffusion_refined" #,"advection"
            , "vector_components"
            , "steady_state", "varying_velocity"
            , "source_stretching"
            ]
for c in POSITIVE:
    run.addCase(".", f"{c}.data", nbProcs=1)

run.printCases()
run.runCases()

## 2. Case A — uniform-field invariance

 With $b\equiv b_0$ and $u=0$ both transport terms vanish identically, so
$b(x,t)$ must stay equal to $b_0$ for all $t$. Machine-precision invariant:
 catches spurious sources, a wrong time factor or bad CL assembly.


In [ ]:
import numpy as np
from trustutils.files import SonFile

t, b = read_son("invariance", "SONDE_CBX")

b0 = 3.0
err = np.max(np.abs(b - b0)) / abs(b0)
print(f"final time       : {t[-1]:.6g}")
print(f"max |b - b0|/b0 : {err:.3e}")
assert err < 1e-10, f"uniform field not conserved (rel. error {err:.3e})"
print("OK: uniform magnetic field is invariant.")

## 3. Case B — decay of a magnetic Fourier mode (+ mesh convergence)

 $b(x,y,0)=\sin(\pi y)$, Dirichlet in $Y$, zero flux in $X$, $u=0$, $\eta=0.05$:

$$b(x,y,t)=\sin(\pi y)\,e^{-\eta\pi^2t}.$$

 We compare the *amplitude ratio* $b(t)/b(0)$ (independent of where the probe
 is snapped) to the analytic factor, over all recorded times.


In [ ]:
ETA, K = 0.05, np.pi
def analytic_ratio(t): return np.exp(-ETA * K**2 * np.asarray(t, float))

TAGS = ["SONDE_Y025", "SONDE_Y050", "SONDE_Y075"]
err_coarse = ratio_error("diffusion", TAGS, analytic_ratio)
err_refine = ratio_error("diffusion_refined", TAGS, analytic_ratio)
print(f"coarse  max |ratio - analytic| : {err_coarse:.4e}")
print(f"refined max |ratio - analytic| : {err_refine:.4e}")

assert err_coarse < 0.05, f"coarse decay error too large: {err_coarse:.3e}"
assert err_refine < err_coarse, "error did not decrease with mesh refinement"
assert err_refine < 0.02, f"refined decay error too large: {err_refine:.3e}"
print("OK: decay matches analytic and converges under refinement.")

# --- Business contract: the transport coefficient is eta ITSELF -------
# The class derives from the ENERGY equation, so the real danger is an
# inherited rho*Cp time factor (~1.2e3 for air). Measure the actual
# relaxation rate lambda = -d/dt ln(b/b0) and compare with eta*pi^2.
t, b = read_son("diffusion", "SONDE_Y050")
lam_measured = -np.log(abs(b[-1] / b[0])) / t[-1]
lam_expected = ETA * np.pi**2
rel = abs(lam_measured - lam_expected) / lam_expected
print(f"measured relaxation rate : {lam_measured:.6f} (expected {lam_expected:.6f})")
print(f"relative error           : {rel:.3e}")
assert rel < 0.05, f"transport coefficient is not eta: rel. error {rel:.3e}"
print("OK: transport coefficient is exactly eta (no rho*Cp leak).")

import matplotlib.pyplot as plt
t, b = read_son("diffusion", "SONDE_Y050")
plt.figure(figsize=(6,4))
plt.plot(t, b/b[0], "o-", ms=2, label="TRUST-LiMe (coarse)")
plt.plot(t, analytic_ratio(t), "k--", label="analytic exp(-eta pi^2 t)")
plt.xlabel("t [s]"); plt.ylabel("b(t)/b(0)"); plt.legend(); plt.grid(True); plt.show()

## 4. Case C — advection of a magnetic bump

 A Gaussian bump advected by a constant $u=(0.2,0)$. The upstream scheme is
 diffusive so the bump broadens, but the peak value must be preserved and
 the field must grow at the advected location $x_0+Ut$.


In [ ]:
t, b_peak0 = read_son("advection", "SONDE_X025")  # initial peak
t2, b_trav  = read_son("advection", "SONDE_X045")  # advected location
peak_pres = np.max(np.abs(b_peak0 - 1.0))
print(f"max |b(t)-b0| at initial peak : {peak_pres:.4e}")
assert peak_pres < 0.05, f"peak not conserved: {peak_pres:.3e}"
print(f"b at x0+Ut : initial {b_trav[0]:.3f} -> final {b_trav[-1]:.3f}")
assert b_trav[0] < 0.3 and b_trav[-1] > 0.5, "bump not transported to the right"
print("OK: bump advected to the right, peak preserved.")

plt.figure(figsize=(6,4))
plt.plot(t, b_peak0, label="b(x0,t)")
plt.plot(t2, b_trav, label="b(x0+Ut,t)")
plt.xlabel("t [s]"); plt.ylabel("b"); plt.legend(); plt.grid(True); plt.show()

## 5. Case D — vector (multi-component) unknown

 `nb_composantes` is omitted, so the default (one component per space direction)
 is used. On the unit square with Dirichlet $b=0$ on all faces, the Laplacian
 eigenmodes $\sin(m\pi x)\sin(n\pi y)$ decay at rate $\eta(m^2+n^2)\pi^2$. The two
 components are given **different** modes so that a component swap or a mixed
 assembly changes the rate and is caught:

$$b_x=\sin(\pi x)\sin(\pi y)e^{-2\eta\pi^2t},\quad b_y=\sin(2\pi x)\sin(\pi y)e^{-5\eta\pi^2t}.$$


In [ ]:
ETA = 0.02
def bx_ratio(t): return np.exp(-2 * ETA * np.pi**2 * np.asarray(t, float))
def by_ratio(t): return np.exp(-5 * ETA * np.pi**2 * np.asarray(t, float))

err_bx = ratio_error("vector_components", ["sonde_bx"], bx_ratio)
err_by = ratio_error("vector_components", ["sonde_by"], by_ratio)
print(f"component 0 (rate 2 eta pi^2) max error : {err_bx:.4e}")
print(f"component 1 (rate 5 eta pi^2) max error : {err_by:.4e}")

# a component swap would make the two rates coincide / mismatch strongly
assert err_bx < 0.03, f"component 0 error too large: {err_bx:.3e}"
assert err_by < 0.03, f"component 1 error too large: {err_by:.3e}"
# sanity: the two rates are genuinely different (guards the test itself)
assert abs(np.exp(-2*ETA*np.pi**2*0.5) - np.exp(-5*ETA*np.pi**2*0.5)) > 0.2
print("OK: components decay independently at their own eigen-rates.")

plt.figure(figsize=(6,4))
t, bx = read_son("vector_components", "sonde_bx")
t, by = read_son("vector_components", "sonde_by")
plt.plot(t, bx/bx[0], label="bx (2 eta pi^2)")
plt.plot(t, by/by[0], label="by (5 eta pi^2)")
plt.xlabel("t [s]"); plt.ylabel("ratio"); plt.legend(); plt.grid(True); plt.show()

## 6. Case E — steady state with a spatially varying diffusivity

 $b=y$ with $\eta(x)=1+x$ and $u=0$. Since $\nabla b=e_y$,
 $\nabla\cdot(\eta\nabla b)=\partial_y(1+x)=0$: the profile is an exact steady
 solution for any diffusivity that varies only in $x$. Any spurious source or
 mis-assembled diffusion operator destroys it, and the diffusivity is read from a
 **functional** field, exercising the non-uniform-$\eta$ path.


In [ ]:
for tag, y_ref in [("SONDE_Y025", 0.25), ("SONDE_Y050", 0.50), ("SONDE_Y075", 0.75)]:
    t, b = read_son("steady_state", tag)
    # compare to the analytic profile value y_ref at every recorded time
    err = np.max(np.abs(b - y_ref))
    print(f"{tag}: b - {y_ref} -> max |err| = {err:.3e}")
    assert err < 1e-8, f"steady state not preserved at {tag}: {err:.3e}"
print("OK: equilibrium with non-uniform eta is preserved.")

plt.figure(figsize=(6,4))
t, b = read_son("steady_state", "SONDE_Y050")
plt.plot(t, b, label="b(x=0.5, y=0.5, t)  [b=y equilibrium]")
plt.axhline(0.5, color="k", ls="--", label="analytic 0.5")
plt.xlabel("t [s]"); plt.ylabel("b"); plt.legend(); plt.grid(True); plt.show()

## 7. Case F — spatially varying velocity

 A strongly varying vortex velocity $u=(-y,x)$ is transferred onto the faces
 from a functional field. With a uniform $b=b_0$, both transport terms vanish, so
 $b$ must stay exactly $b_0$. This proves the varying-velocity path is correctly
 instantiated and copied onto the face-centred unknown.


In [ ]:
t, b = read_son("varying_velocity", "SONDE_CBX")
b0 = 3.0
err = np.max(np.abs(b - b0)) / abs(b0)
print(f"max |b - b0|/b0 : {err:.3e}")
assert err < 1e-10, f"varying-velocity invariance broken (rel. error {err:.3e})"
print("OK: uniform field invariant even under a varying velocity.")

## 8. Case G — the M1 induction source term $S=(b\cdot\nabla)u$

 The `sources { induction_stretching { } }` block adds $S=(b\cdot\nabla)u$ to
 the right-hand side. With a uniform $b=(B_0,0)$ and $u=(x,0)$ (so
 $\nabla u=[[1,0],[0,0]]$), convection and diffusion vanish identically and

$$\frac{d b_0}{dt}=B_0\frac{\partial u_0}{\partial x}=B_0,\qquad \frac{d b_1}{dt}=0,$$

 i.e. $b_0(t)=B_0e^{t}$ and $b_1\equiv 0$. A wrong factor, a transposed
 gradient index, or a dropped term changes the exponential rate, so this is a
 sharp absolute check of the source magnitude and sign.


In [ ]:
import numpy as np
t, b0 = read_son("source_stretching", "sonde_b0")
t, b1 = read_son("source_stretching", "sonde_b1")
b0_0 = b0[0]
expected = b0_0 * np.exp(t)
err = np.max(np.abs(b0 - expected)) / abs(b0_0)
print(f"b0(0)={b0_0:.4f}  b0(t_end)={b0[-1]:.4f}  expected exp(t_end)={expected[-1]:.4f}")
print(f"max |b0 - B0 exp(t)| / B0 : {err:.3e}")
# the source must reproduce exp(t) to a few percent (first-order in dt
# for an explicit term; dt=2e-3 keeps the error well below 1%)
assert err < 0.01, f"induction source rate wrong: {err:.3e}"
# component 1 has zero source and must stay at zero
err1 = np.max(np.abs(b1))
print(f"max |b1| (should stay 0)   : {err1:.3e}")
assert err1 < 1e-8, f"b1 should remain null: {err1:.3e}"
print("OK: M1 source S=(b.grad)u has the correct magnitude, sign and structure.")

plt.figure(figsize=(6,4))
plt.plot(t, b0 / b0_0, label="b0/B0 (simulated)")
plt.plot(t, np.exp(t), "k--", label="exp(t) (analytic)")
plt.xlabel("t [s]"); plt.ylabel("ratio"); plt.legend(); plt.grid(True); plt.show()

## 9. Negative cases — malformed inputs must fail cleanly

 These are run **directly** with the `trust` command (not through `runCases`,
 which would abort the notebook). Each must terminate with a non-zero exit code
 and a readable diagnostic, rather than crashing silently.


In [ ]:
import subprocess, tempfile, shutil, os, glob

NEGATIVE = ["error_missing_eta", "error_bad_nbcomposantes", "error_bad_bc"]

def run_negative(case):
    tmp = tempfile.mkdtemp(prefix=f"lime_{case}_")
    shutil.copy(os.path.join("src", f"{case}.data"), tmp)
    try:
        p = subprocess.run(["trust", "-partition", case],
                           cwd=tmp, capture_output=True, text=True, timeout=300)
        rc, out = p.returncode, (p.stdout + p.stderr)
    finally:
        shutil.rmtree(tmp, ignore_errors=True)
    # a clean TRUST failure prints a diagnostic and exits non-zero
    assert rc != 0, f"{case}: expected a non-zero exit, got {rc}"
    assert out.strip(), f"{case}: no diagnostic message emitted"
    return rc, out

for case in NEGATIVE:
    rc, out = run_negative(case)
    tail = [l for l in out.splitlines() if l.strip()][-1:] or [""]
    print(f"{case}: rc={rc}  |  {tail[0][:90]}")
print("OK: all malformed inputs rejected with a readable message.")

## Summary of the business-logic contracts checked above

| # | Modelling decision | Status |
|---|---|:--:|
| 1 | transport coefficient is $\eta$ itself — **no** $\rho C_p$ leak from the energy base class | OK: measured rate vs $\eta\pi^2$ |
| 2 | $\eta$ used verbatim as diffusion coefficient | OK: decay rate |
| 4 | components transported independently, no coupling | OK: two distinct eigen-rates |
| 5 | no spurious source (equilibria preserved) | OK: 1e-10 / 1e-8 |
| 6 | diffusion in divergence form $\nabla\cdot(\eta\nabla b)$ | OK: eta=1+x equilibrium |
| 7 | velocity is given data, frozen | OK: advection + varying velocity |
| 8 | Dirichlet prescribes b; Neumann = zero flux | OK: implied by the analytic rate |
| 11 | malformed input fails cleanly | OK: 3 negative cases |
| 10 | Gauss law $\nabla\cdot b = 0$ | **NOT enforced, NOT testable in 2D** |
| 9 | conducting-wall BC (interface current) | out of scope, by design |

The Gauss law (row 10) is the one physics-level gap: in 2D a divergence-free
field vanishing on the whole boundary is identically zero, so an exact
non-trivial test would need per-component boundary conditions, which TRUST
scalar CLs do not provide. It requires the deferred M3 formulation.
See `KERNEL_REUSE_NOTES.md`, section "Business-logic contracts".


## Computer Performance


In [ ]:
run.tablePerf()